# Batch MedSAM2 TIC pipeline

For every row of the master CSV (the case list; it is only read): locate the case's `*_BMODE.nii`, `*_CEUS.nii` and `*-MC_VOI.nii.gz` in its `Data Dir`, compute six TICs, fit each one (log-normal + wash rates, through the QuantUS framework via `medsam2_tic.py`), write per-case files to `{TIC curve save path or DEST_ROOT}/{case_key}/`, and write the parameters to `RESULTS_XLSX`, **one tab per method**.

| Method / tab | ROI | Motion |
|---|---|---|
| `3D_No_MC` | drawn 3D VOI | none |
| `3D_MC` | drawn 3D VOI | shifted by the 3D motion compensation |
| `3D_Sam2` | MedSAM2 3D mask (adaptive bbox) | on each frame's tracked 3D bbox |
| `2D_original_ROI` | drawn VOI's slice at `z_ref` | none |
| `2D_sam2_fixed_frame` | MedSAM2 on the `z_ref` plane | reference bbox X/Y every frame |
| `2D_sam2_MC_frame` | MedSAM2 on the `z_ref` plane | each frame's tracked bbox X/Y |

`z_ref` is the centre slice of the reference frame's bbox (`motion_compensation.reference_bbox`) and never moves: a sonographer's 2D probe cannot follow the lesion in elevation. The 2D arms match `MedSam2_2D_inference.ipynb`.

Each tab has the case's Site / Patient Number / Visit / Bolus / case_key, then `{method}_{AUC, PE, TP, MTT, T0, Mu, Sigma, R2, roughness, snr, WashIn, WashOut}` plus `{method}_volume_mm3` (3D) or `{method}_area_mm2` (2D), the ROI size on its first non-empty frame. TICs are decompressed (`raw * 3e4/255 + 3e4`) before fitting.

In [8]:
%reload_ext autoreload
%autoreload 2

import os
import sys
import glob
import fnmatch
import shutil
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # batch: render to file, no GUI windows
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

# The notebook lives in CLI-Demos; `src` is one level up.
if not Path("src").exists() and Path("../src").exists():
    os.chdir(Path.cwd().parent)
for _p in ("CLI-Demos", os.path.join("CLI-Demos", "medsam2_function")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from src.entrypoints import scan_loading_step, seg_loading_step
from medsam2_3d_mask import Medsam2AdaptiveBboxMasker
from medsam2_viewer_base import run_medsam2_2d
from fold_checkpoints import checkpoint_for
from medsam2_tic import make_analysis, lognormal_quantification, fitted_curve, fit_quality, cropped_masks
print("CWD:", Path.cwd())

CWD: /home/ahmed-el-kaffas/Documents/Github/QuantUS/engines/ceus


## 1. Configuration

In [9]:
MASTER_CSV = "/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/MedSAM2_TIC_results/MedSAM2_5fold_TIC_results.csv"
DEST_ROOT  = "/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/MedSAM2_TIC_results"   # used when a row has no "TIC curve save path"
RESULTS_XLSX = MASTER_CSV.replace(".csv", ".xlsx")   # results, one tab per method

# Each case uses the checkpoint of the fold that held its patient out (fold_checkpoints.py);
# set MEDSAM2_CHECKPOINT to force one checkpoint for every case.
MEDSAM2_CHECKPOINT_OVERRIDE = os.environ.get("MEDSAM2_CHECKPOINT")
MODEL_CFG = "configs/sam2.1_hiera_t512.yaml"
DEVICE    = "cuda"

SMOOTH_SIGMA_MM   = 1.0    # 3D Gaussian over the MedSAM2 3D mask; 0 disables
SYNTHETIC_CORONAL = False  # True = model X(z) from centre-axial + sagittal instead of a coronal guide slice

USE_CUPY            = True   # CuPy masked mean for the full-volume `sam2` TIC; falls back to NumPy
N_FRAMES_TO_ANALYZE = None   # wash-in/wash-out window, in valid frames; None = the whole curve

ONLY_CASES = None   # e.g. ["UCSD-P05-V01-CE1"]; None = every row
OVERWRITE  = True   # with ONLY_CASES = None: start RESULTS_XLSX from empty tabs. False = keep it and skip finished cases

if MEDSAM2_CHECKPOINT_OVERRIDE:
    assert os.path.isfile(MEDSAM2_CHECKPOINT_OVERRIDE), f"checkpoint not found: {MEDSAM2_CHECKPOINT_OVERRIDE}"

## 2. File discovery

Files are matched case-insensitively on the `Site-Patient-Visit-Bolus` stem, with a looser `*{visit}*{bolus}*` fallback for the `P010`-vs-`P10` quirks in the CSV.

In [10]:
def _clean(s):
    return str(s).strip()

def make_case_key(site, patient, visit, bolus):
    return f"{_clean(site)}-{_clean(patient)}-{_clean(visit)}-{_clean(bolus)}"

def _find_one(data_dir, stem, patterns, label):
    """First file under data_dir (recursive) whose name starts with `stem` and ends with a pattern,
    or, with an empty stem, fnmatch-es a pattern. Case-insensitive."""
    data_dir = _clean(data_dir)
    if not os.path.isdir(data_dir):
        print(f"    [warn] data dir does not exist: {data_dir}")
        return None
    names = [(fp, os.path.basename(fp).lower())
             for fp in glob.glob(os.path.join(data_dir, "**", "*"), recursive=True) if os.path.isfile(fp)]
    pats = [p.lower() for p in patterns]
    def _rank(n):   # index of the first pattern this name matches, or None
        return next((i for i, pat in enumerate(pats)
                     if (n.startswith(stem.lower()) and n.endswith(pat) if stem else fnmatch.fnmatch(n, pat))), None)
    # shallowest first (data_dir beats e.g. ArchiveVOI/), then pattern order
    hits = sorted((fp.count(os.sep), r, fp) for fp, n in names if (r := _rank(n)) is not None)
    if not hits:
        return None
    if len(hits) > 1:
        print(f"    [warn] multiple {label} matches for '{stem or pats[0]}', using {os.path.basename(hits[0][2])}")
    return hits[0][2]

def locate_inputs(row):
    """BMODE / CEUS / MC_VOI paths for a CSV row (None where not found)."""
    data_dir, visit, bolus = row["Data Dir"], _clean(row["Visit"]), _clean(row["Bolus"])
    stem = make_case_key(row["Site"], row["Patient Number"], visit, bolus)
    found = {}
    for key, suffixes, loose in (("bmode", ["_BMODE.nii", "BMODE.nii"], f"*{visit}*{bolus}*_BMODE.nii"),
                                 ("ceus",  ["_CEUS.nii", "CEUS.nii"],   f"*{visit}*{bolus}*_CEUS.nii"),
                                 ("seg",   ["-MC_VOI.nii.gz", "MC_VOI.nii.gz","MC_VOI_2.nii.gz"], f"*{visit}*{bolus}*MC_VOI*.nii.gz")):
        found[key] = _find_one(data_dir, stem, suffixes, key) or _find_one(data_dir, "", [loose], f"{key} (loose)")
    return found

## 3. TICs for one case

One pass over the frames scores all six ROIs. `3D_No_MC`/`3D_MC` are scored on a crop around the VOI (`cropped_masks`, identical to `apply_to_mask` on the full volume), the 2D arms on the `z_ref` plane. The 2D MedSAM2 calls reuse the 3D masker's image predictor, so both use the same checkpoint.

In [11]:
METHODS_3D = ("3D_No_MC", "3D_MC", "3D_Sam2")
METHODS_2D = ("2D_original_ROI", "2D_sam2_fixed_frame", "2D_sam2_MC_frame")
METHODS    = METHODS_3D + METHODS_2D
SIZE_COL   = {m: "volume_mm3" if m in METHODS_3D else "area_mm2" for m in METHODS}

def _box_xy(bbox):
    return [int(v) for v in (bbox.x_min, bbox.y_min, bbox.x_max, bbox.y_max)]

def _sam2_2d(predictor, plane_yx, box):
    m = run_medsam2_2d(predictor, plane_yx, box, device=DEVICE)
    return np.zeros(plane_yx.shape, bool) if m is None else m

def _decompress(tic):
    return np.asarray(tic, dtype=float) * 3e4 / 255.0 + 3e4

def compute_all_tics(image_data, bmode_image_data, seg_data, masker, frame_start=None, frame_end=None):
    """Raw TIC + ROI size per method over frames [frame_start, frame_end) (None = whole acquisition)."""
    n_frames = bmode_image_data.pixel_data.shape[-1]
    frames = np.arange(max(0, frame_start or 0), min(n_frames, frame_end or n_frames))

    mc = seg_data.motion_compensation
    ref = mc.reference_bbox
    z_ref = int((ref.z_min + ref.z_max) // 2)
    box_ref = _box_xy(ref)
    roi_2d = seg_data.seg_mask[:, :, z_ref].T > 0                        # (Y, X), as the 2D planes
    region, nomc_crop, mc_crop_for = cropped_masks(seg_data)

    ceus, bmode = image_data.intensities_for_analysis, bmode_image_data.pixel_data
    voxel_mm3 = float(np.prod(image_data.pixdim))
    pixel_mm2 = float(image_data.pixdim[0] * image_data.pixdim[1])
    analyses = {m: make_analysis(image_data, seg_data, frames, USE_CUPY and m == "3D_Sam2") for m in METHODS}
    sizes = {m: [] for m in METHODS}

    for pos, t in enumerate(tqdm(frames, desc="  TIC frames", leave=False)):
        volume = ceus[:, :, :, t]
        plane = volume[:, :, z_ref].T
        bmode_plane = np.asarray(bmode[:, :, z_ref, t], dtype=np.uint8).T
        sam2_mask = masker.compute_frame(t)["mask_3d"]
        masker.cache.clear()           # each entry holds a full mask + volume; every frame is visited once
        scored = {
            "3D_No_MC":            (volume[region], nomc_crop),
            "3D_MC":               (volume[region], mc_crop_for(t)),
            "3D_Sam2":             (volume, sam2_mask),
            "2D_original_ROI":     (plane, roi_2d),
            "2D_sam2_fixed_frame": (plane, _sam2_2d(masker.image_predictor, bmode_plane, box_ref)),
            "2D_sam2_MC_frame":    (plane, _sam2_2d(masker.image_predictor, bmode_plane, _box_xy(mc.tracked_bboxes[t]))),
        }
        for m, (frame, mask) in scored.items():
            analyses[m].extract_frame_features(frame, mask, pos)
            sizes[m].append(np.count_nonzero(mask) * (voxel_mm3 if m in METHODS_3D else pixel_mm2))

    has_rate = getattr(image_data, "frame_rate", 0) > 0
    out = {"frames": frames, "z_ref": z_ref, "analyses": analyses,
           "time_axis": frames * image_data.frame_rate if has_rate else frames.astype(float),
           "x_label": "Time (s)" if has_rate else "Frame index"}
    for m in METHODS:
        out[f"tic_{m}"] = _decompress(analyses[m].curves[0]["TIC"])
        analyses[m].curves[0]["TIC"] = out[f"tic_{m}"].tolist()       # the fit runs on the decompressed curve
        out[f"size_{m}"] = np.array(sizes[m])
    return out

## 4. Per-case driver

Fits every TIC, writes `{case_key}_tic_curve.csv` (raw + fitted curve and ROI size per frame), `{case_key}_fit_params.csv` (one row per method) and `{case_key}_tic_plot.png`, and returns the row's CSV values.

In [12]:
STYLE = {"3D_No_MC": "green", "3D_MC": "red", "3D_Sam2": "blue",
         "2D_original_ROI": "olive", "2D_sam2_fixed_frame": "orange", "2D_sam2_MC_frame": "purple"}

def _first_nonzero(sizes):
    pos = sizes[sizes > 0]
    return float(pos[0]) if pos.size else 0.0

def _save_case_outputs(case_dir, case_key, tic_data, quants, metrics, fitted):
    os.makedirs(case_dir, exist_ok=True)
    t = tic_data["time_axis"]

    cols = {"frame": tic_data["frames"], "time": t}
    for m in METHODS:
        cols.update({f"tic_{m}_raw": tic_data[f"tic_{m}"], f"tic_{m}_fit": fitted[m],
                     f"{SIZE_COL[m]}_{m}": tic_data[f"size_{m}"]})
    paths = {"tic_curve": os.path.join(case_dir, f"{case_key}_tic_curve.csv"),
             "fit_params": os.path.join(case_dir, f"{case_key}_fit_params.csv"),
             "plot": os.path.join(case_dir, f"{case_key}_tic_plot.png")}
    pd.DataFrame(cols).to_csv(paths["tic_curve"], index=False)

    rows = []
    for m in METHODS:
        d = quants[m].data_dict[0]
        rows.append({"method": m,
                     **{k.lower(): d[f"{k}_full_TIC"] for k in ("AUC", "PE", "TP", "MTT", "T0", "Mu", "Sigma")},
                     "pe_loc": d["PE_Ix_full_TIC"], "baseline": float(np.nanmin(tic_data[f"tic_{m}"])),
                     **{k: metrics[m][k] for k in ("r2", "roughness", "snr", "wash_in", "wash_out")},
                     "size": _first_nonzero(tic_data[f"size_{m}"]), "size_unit": SIZE_COL[m]})
    pd.DataFrame(rows).to_csv(paths["fit_params"], index=False)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)
    for ax, group, title in ((axes[0], METHODS_3D, "3D"), (axes[1], METHODS_2D, f"2D, z = {tic_data['z_ref']}")):
        for m in group:
            ax.plot(t, tic_data[f"tic_{m}"], color=STYLE[m], lw=2, alpha=0.35, label=f"{m} raw")
            ax.plot(t, fitted[m], color=STYLE[m], lw=2, ls="--", label=f"{m} fit")
        ax.set_xlabel(tic_data["x_label"]); ax.set_ylabel("Mean intensity")
        ax.set_title(f"{case_key}  {title}"); ax.legend(loc="upper right", fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(paths["plot"], dpi=120); plt.close(fig)
    return paths

def process_case(row, frame_start=None, frame_end=None):
    """Full pipeline for one CSV row; returns {"status": ..., **CSV values}."""
    case_key = make_case_key(row["Site"], row["Patient Number"], row["Visit"], row["Bolus"])
    print(f"\n=== {case_key} ===")
    # fail before loading any volume if there is no held-out checkpoint for this patient
    checkpoint = MEDSAM2_CHECKPOINT_OVERRIDE or checkpoint_for(row["Site"], row["Patient Number"])

    inputs = locate_inputs(row)
    missing = [k for k, v in inputs.items() if not v]
    if missing:
        print(f"  [SKIP] missing inputs: {missing}")
        return {"case_key": case_key, "status": f"missing:{','.join(missing)}"}
    for k, v in inputs.items():
        print(f"  {k:5s}: {os.path.basename(v)}")

    image_data       = scan_loading_step("nifti", inputs["ceus"])
    bmode_image_data = scan_loading_step("nifti", inputs["bmode"])
    seg_data         = seg_loading_step("nifti", image_data, inputs["seg"], inputs["ceus"])

    masker = Medsam2AdaptiveBboxMasker(seg_data, bmode_image_data, MODEL_CFG, checkpoint, device=DEVICE,
                                       smooth_sigma_mm=SMOOTH_SIGMA_MM, synthetic_coronal=SYNTHETIC_CORONAL)
    started = time.time()
    try:
        tic_data = compute_all_tics(image_data, bmode_image_data, seg_data, masker, frame_start, frame_end)
    finally:
        del masker
        torch.cuda.empty_cache()
    print(f"  TICs computed in {time.time() - started:.0f}s  (z_ref = {tic_data['z_ref']})")

    quants  = {m: lognormal_quantification(tic_data["analyses"][m], N_FRAMES_TO_ANALYZE) for m in METHODS}
    metrics = {m: fit_quality(tic_data["analyses"][m], quants[m]) for m in METHODS}
    fitted  = {m: fitted_curve(tic_data["analyses"][m], quants[m]) for m in METHODS}

    case_dir = os.path.join(_clean(row.get("TIC curve save path", "")) or DEST_ROOT, case_key)
    _save_case_outputs(case_dir, case_key, tic_data, quants, metrics, fitted)
    print(f"  wrote: {case_dir}")

    result = {"case_key": case_key, "status": "ok", "checkpoint": os.path.basename(checkpoint)}
    for m in METHODS:
        d, met = quants[m].data_dict[0], metrics[m]
        result.update({f"{m}_{k}": d[f"{k}_full_TIC"] for k in ("AUC", "PE", "TP", "MTT", "T0", "Mu", "Sigma")})
        result.update({f"{m}_R2": met["r2"], f"{m}_roughness": met["roughness"], f"{m}_snr": met["snr"],
                       f"{m}_WashIn": d["WashInRate_TIC"], f"{m}_WashOut": d["WashOutRate_TIC"],
                       f"{m}_{SIZE_COL[m]}": _first_nonzero(tic_data[f"size_{m}"])})
    return result

## 5. Batch run

Results go to `RESULTS_XLSX`, one tab per method, rewritten after **every** case so a crash keeps the finished rows. An existing workbook is first backed up (timestamped). A full run (`ONLY_CASES = None`, `OVERWRITE = True`) starts from empty tabs, so no old value survives a failed case; a test on `ONLY_CASES`, or `OVERWRITE = False`, keeps the other rows. The master CSV is not modified.

⚠️ The 3D MedSAM2 mask is one SAM2 pass per axial slice per frame, so the full CSV is an overnight job. Try one case first with `ONLY_CASES`.

In [13]:
df = pd.read_csv(MASTER_CSV)
df["Site"] = df["Site"].astype(str).str.strip()
df["case_key"] = [make_case_key(r["Site"], r["Patient Number"], r["Visit"], r["Bolus"]) for _, r in df.iterrows()]

PARAM_COLS = ["AUC", "PE", "TP", "MTT", "T0", "Mu", "Sigma", "R2", "roughness", "snr", "WashIn", "WashOut"]
ID_COLS = ["Site", "Patient Number", "Visit", "Bolus", "case_key"]
TAB_COLS = {m: [f"{m}_{c}" for c in (*PARAM_COLS, SIZE_COL[m])] + (["checkpoint"] if "sam2" in m.lower() else [])
            for m in METHODS}

def read_tabs(keep_existing):
    """One empty row per case in every tab; with keep_existing, filled from RESULTS_XLSX where it has the case."""
    tabs = {}
    for m, cols in TAB_COLS.items():
        tabs[m] = df[ID_COLS].copy()
        for c in cols:
            tabs[m][c] = None if c == "checkpoint" else np.nan
    if keep_existing and os.path.exists(RESULTS_XLSX):
        for m, prev in pd.read_excel(RESULTS_XLSX, sheet_name=None).items():
            if m in tabs:
                prev = prev.drop_duplicates("case_key").set_index("case_key")
                for col in prev.columns.intersection(TAB_COLS[m]):
                    tabs[m][col] = tabs[m]["case_key"].map(prev[col])
    return tabs

if os.path.exists(RESULTS_XLSX):
    backup = RESULTS_XLSX.replace(".xlsx", f"_backup_{time.strftime('%Y%m%d_%H%M%S')}.xlsx")
    shutil.copy2(RESULTS_XLSX, backup)
    print(f"backed up existing results -> {backup}")
# a full re-run starts from empty tabs; a test on ONLY_CASES or a resume keeps the other rows
tabs = read_tabs(keep_existing=not (OVERWRITE and ONLY_CASES is None))

def save_tabs():
    with pd.ExcelWriter(RESULTS_XLSX) as writer:
        for m in METHODS:
            tabs[m].to_excel(writer, sheet_name=m, index=False)

def write_back(idx, result):
    for m in METHODS:
        for col in TAB_COLS[m]:
            if col in result:
                tabs[m].at[idx, col] = result[col]
    save_tabs()
print(f"{len(df)} cases -> {RESULTS_XLSX}  (tabs: {', '.join(METHODS)})")

backed up existing results -> /media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/MedSAM2_TIC_results/MedSAM2_5fold_TIC_results_backup_20261104_201351.xlsx
41 cases -> /media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/MedSAM2_TIC_results/MedSAM2_5fold_TIC_results.xlsx  (tabs: 3D_No_MC, 3D_MC, 3D_Sam2, 2D_original_ROI, 2D_sam2_fixed_frame, 2D_sam2_MC_frame)


In [ ]:
summary = []
for idx, row in df.iterrows():
    case_key = row["case_key"]
    if ONLY_CASES is not None and case_key not in ONLY_CASES:
        continue
    if not OVERWRITE and pd.notna(tabs["3D_Sam2"].at[idx, "3D_Sam2_R2"]):
        summary.append({"case_key": case_key, "status": "skipped-done"})
        continue
    try:
        result = process_case(row)
    except Exception as e:
        traceback.print_exc()
        result = {"case_key": case_key, "status": f"error:{e}"}
    summary.append(result)
    if result["status"] == "ok":
        write_back(idx, result)
        print(f"  results updated for {case_key}")

summary_df = pd.DataFrame(summary)
print(summary_df["status"].value_counts().to_string() if not summary_df.empty else "nothing run")

## 6. (Optional) Re-run one case on a frame window

For acquisitions with bad frames: TICs over `[FRAME_START, FRAME_END)` only (`None` = acquisition boundary). Per-case files and this case's row in every tab are overwritten; the other cases' rows are re-read from `RESULTS_XLSX` first, so they are kept whatever `ONLY_CASES`/`OVERWRITE` were. Run section 5's first cell before this one.

In [17]:
df

,Site,Patient Number,Visit,Bolus,UseCheckPoint,Data Dir,TIC curve save path,case_key
0,UCSD,P05,V01,CE1,4,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,UCSD-P05-V01-CE1
1,UCSD,P05,V01,CE2,4,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,UCSD-P05-V01-CE2
2,UCSD,P05,V02,CE1,4,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,UCSD-P05-V02-CE1
3,UCSD,P05,V02,CE2,4,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,UCSD-P05-V02-CE2
4,UCSD,P05,V03,CE1,4,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,UCSD-P05-V03-CE1
5,UCSD,P05,V03,CE2,4,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,UCSD-P05-V03-CE2
6,UCSD,P07,V01,CE1RUN2,5,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,UCSD-P07-V01-CE1RUN2
7,UCSD,P07,V01,CE2,5,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,UCSD-P07-V01-CE2
8,UCSD,P07,V02,CE01RUN2,5,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,UCSD-P07-V02-CE01RUN2
9,UCSD,P07,V02,CE2,5,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,UCSD-P07-V02-CE2


In [ ]:
RERUN_ROW_IDX = 34
FRAME_START, FRAME_END = 0, 510

res = process_case(df.iloc[RERUN_ROW_IDX], FRAME_START, FRAME_END)
if res["status"] == "ok":
    tabs = read_tabs(keep_existing=True)      # every other case's rows come from the workbook on disk
    write_back(df.index[RERUN_ROW_IDX], res)
res


=== TJU-P014-V01-CEUS2 ===
TJU-P014: fold 2 (held out) -> fold_2.pt
    [warn] multiple seg (loose) matches for '*v01*ceus2*mc_voi*.nii.gz', using TJUH-P014-V01-CEUS2-MC_VOI_2.nii.gz
  bmode: TJUH-P014-V01-CEUS2_10.43.42_mf_sip_capture_50_2_1_0_BMODE.nii
  ceus : TJUH-P014-V01-CEUS2_10.43.42_mf_sip_capture_50_2_1_0_CEUS.nii
  seg  : TJUH-P014-V01-CEUS2-MC_VOI_2.nii.gz


  TICs computed in 1057s  (z_ref = 159)
  wrote: /media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/MedSAM2_TIC_results/TJU-P014-V01-CEUS2


{'case_key': 'TJU-P014-V01-CEUS2',
 'status': 'ok',
 'checkpoint': 'fold_2.pt',
 '3D_No_MC_AUC': np.float64(317930.6321183083),
 '3D_No_MC_PE': np.float64(1673.4230882511195),
 '3D_No_MC_TP': np.float64(31.220888056096683),
 '3D_No_MC_MTT': np.float64(263.4622544203957),
 '3D_No_MC_T0': np.float64(1.9209186686878919),
 '3D_No_MC_Mu': np.float64(4.862969193754487),
 '3D_No_MC_Sigma': np.float64(1.1924268673463219),
 '3D_No_MC_R2': np.float64(0.878023578708472),
 '3D_No_MC_roughness': np.float64(100.78679690219529),
 '3D_No_MC_snr': np.float64(54.33593091540871),
 '3D_No_MC_WashIn': np.float64(30.97212573505541),
 '3D_No_MC_WashOut': np.float64(-4.000747489870748),
 '3D_No_MC_volume_mm3': 139254.0689123422,
 '3D_MC_AUC': np.float64(319979.7963976565),
 '3D_MC_PE': np.float64(1706.884520294567),
 '3D_MC_TP': np.float64(29.386289534970118),
 '3D_MC_MTT': np.float64(269.64948819865896),
 '3D_MC_T0': np.float64(2.181683890523804),
 '3D_MC_Mu': np.float64(4.8582580235821),
 '3D_MC_Sigma': np.